# SMS Spam Detection - Recurrent Neural Network (LSTM/GRU) Pipeline
### 3MTT NextGen Cohort - Step 54 Evaluation

### 1. Discovery Phase (Data Preparation & Label Mapping)
Loading the SMS messaging dataset, mapping text labels to binary targets, splitting datasets, and applying tokenization and padding sequences.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_score

# Locate and load the target dataset
csv_file = [f for f in os.listdir('.') if f.endswith('.csv')][0]
# Read handling potential encoding issues common with text datasets
try:
    df = pd.read_csv(csv_file, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(csv_file, encoding='latin-1')

df.columns = [c.strip() for c in df.columns]

# Identify columns dynamically
text_col = [c for c in df.columns if 'text' in c.lower() or 'sms' in c.lower() or 'v2' in c.lower()][0]
label_col = [c for c in df.columns if 'label' in c.lower() or 'v1' in c.lower()][0]

df = df.dropna(subset=[text_col, label_col])

# Load the SMS dataset and map labels: 'spam' -> 1, 'ham' -> 0
df['target'] = df[label_col].apply(lambda x: 1 if str(x).strip().lower() == 'spam' else 0)

X_text = df[text_col].astype(str).values
y_labels = df['target'].values

# Split data into training and validation sets for unbiased model evaluation
X_train_text, X_val_text, y_train, y_val = train_test_split(
    X_text, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

# Tokenize text into integer sequences using Keras Tokenizer
max_words = 5000
max_len = 50
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_val_seq = tokenizer.texts_to_sequences(X_val_text)

# Apply 'pad_sequences' to standardize all inputs to a fixed length (e.g., 50 tokens)
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding='post', truncating='post')

print("--- Structural Dimensions Matrix ---")
print(f"Padded Training Set Shape: {X_train_pad.shape}")
print(f"Padded Validation Set Shape: {X_val_pad.shape}")

### 2. Technical Phase (RNN/LSTM Architecture & Compilation)
Building a sequential deep learning framework featuring dense embeddings, recurrent gates, and early-stopping constraints.

In [ ]:
tf.random.set_seed(42)
np.random.seed(42)

# Build a Keras Sequential model with an Embedding layer, an LSTM/GRU layer, and regularizations
model = Sequential([
    Embedding(input_dim=max_words, output_dim=32, input_length=max_len, name="Embedding_Layer"),
    # Add an LSTM or GRU layer to capture sequential word dependencies and mitigate vanishing gradients
    LSTM(32, return_sequences=False, name="LSTM_Layer"),
    # Include a 'Dropout(0.2)' layer to prevent memorization of specific spam patterns
    Dropout(0.2, name="Dropout_Regularization"),
    Dense(16, activation='relu', name="Dense_Hidden_Layer"),
    Dense(1, activation='sigmoid', name="Output_Layer")
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Configure the 'EarlyStopping' callback to monitor 'val_loss' and halt training at optimal convergence
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

# Train the model
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

### 3. Model Diagnostic Curves
Plotting Training/Validation loss curves to diagnose fit and verify stable, non-overfitting optimization tracks.

In [ ]:
# Plot Training/Validation loss curves to diagnose fit and overfitting
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Training Loss', marker='o', color='royalblue')
plt.plot(history.history['val_loss'], label='Validation Loss', marker='s', color='crimson')
plt.title('RNN Loss Convergence Evaluation')
plt.xlabel('Epochs')
plt.ylabel('Binary Crossentropy')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

### 4. Action Phase (Triage Case Execution & Evaluation Performance Logs)
Executing verification requests on out-of-sample diagnostic text strings and analyzing precision boundaries.

In [ ]:
# Evaluate performance parameters across standard validation sets
y_pred_probs = model.predict(X_val_pad)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

print("--- Validation Confusion Matrix ---")
print(confusion_matrix(y_val, y_pred))
print("\n--- Detailed Performance Classification Report ---")
print(classification_report(y_val, y_pred, target_names=['Ham (0)', 'Spam (1)']))

# Perform a triage test by passing three specific messages through your trained model
triage_messages = [
    "Hey, are we still meeting for lunch?",
    "URGENT! Your account is locked. Click here to verify.",
    "Congratulations, you won a $500 gift card!"
]

triage_seqs = tokenizer.texts_to_sequences(triage_messages)
triage_pad = pad_sequences(triage_seqs, maxlen=max_len, padding='post', truncating='post')
triage_predictions = model.predict(triage_pad).flatten()

print("\n--- Action Phase Triage Test Case Logs ---")
for msg, prob in zip(triage_messages, triage_predictions):
    print(f"\nMessage Body: \"{msg}\"")
    print(f"Calculated Spam Probability: {prob:.4f}")
    print(f"Assigned Classification: {'SPAM (1)' if prob > 0.5 else 'HAM (0)'}")

### 5. Technical Framework & Strategic UX Recommendation Report

#### Rationale for Sequence Padding Boundaries
SMS texts are short but vary in length. For vector processing within deep learning architectures, feature inputs must be structured into identical dimensions. Setting the sequence length at a maximum of 50 tokens via `pad_sequences` covers the majority of typical SMS message sizes. Shorter texts are cleanly padded with zeros at the end (`padding='post'`), ensuring uniform network mapping without discarding essential sentence context.

#### Mitigation of Vanishing Gradients via Recurrent Gates
Traditional Recurrent Neural Networks struggle with vanishing gradient drops during backpropagation over longer text strings, which limits their ability to capture historical context. The `LSTM` layer counteracts this by using a cell state regulated by input, forget, and output gates. These gates allow errors to flow smoothly across time slices, allowing the network to retain text patterns even across long sequences.

#### Rationale for Prioritizing Precision over Accuracy in Spam Filtering
In mobile communication systems, evaluation costs are highly asymmetric:
- **False Positives (The Big Risk):** Classifying a legitimate message (Ham) as Spam. This means an important communication (e.g., a bank alert, flight cancellation notice, or family emergency text) gets incorrectly moved to a junk folder, which directly hurts the user experience.
- **False Negatives (Minor Overhead):** Allowing an occasional spam text to land in the main inbox. This is slightly annoying, but the user can easily ignore or manually delete it, causing zero critical information loss.
- **Strategic Recommendation:** We prioritize high **Precision** over broad accuracy metrics. By raising our confidence threshold to **95%** before automatically moving a message to junk routing, we ensure that important, legitimate user communications are almost never blocked, maintaining a reliable user experience.